# A2A-NT: Results Analysis

Replicates the metrics from Wang et al. (2025):
- **Price Reduction Rate (PRR)**: how much the buyer drove the price down from retail
- **Relative Profit (RP)**: seller profit relative to a baseline seller
- **Deal Rate**: fraction of negotiations that ended in acceptance
- **Anomaly rates**: overpayment, deadlock, constraint violation

In [ ]:
import json
import os
import glob
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

RESULTS_DIR = "../results"  # adjust if needed
print(f"Looking for results in: {os.path.abspath(RESULTS_DIR)}")

In [ ]:
# ── Load all JSON result files ──────────────────────────────────────────────

records = []
for path in glob.glob(os.path.join(RESULTS_DIR, "**/*.json"), recursive=True):
    with open(path) as f:
        data = json.load(f)

    retail    = float(data["product_data"]["Retail Price"].replace("$","").replace(",",""))
    wholesale = float(data["product_data"]["Wholesale Price"].replace("$","").replace(",",""))
    final_price = data["seller_price_offers"][-1] if data["seller_price_offers"] else retail
    result = data.get("negotiation_result", "unknown")
    accepted = result == "accepted"

    # Price Reduction Rate (buyer perspective): how much below retail did they land?
    prr_buyer  = (retail - final_price) / retail * 100 if accepted else None
    # Price Reduction Rate (seller perspective): retail minus final, as % of retail
    prr_seller = (retail - final_price) / retail * 100 if accepted else None

    seller_profit  = final_price - wholesale if accepted else None
    overpayment    = accepted and final_price > retail
    deadlock       = result == "max_turns_reached"
    constraint_vio = accepted and final_price < wholesale

    records.append({
        "file":           path,
        "product_id":     data["product_id"],
        "product_name":   data["product_data"]["Product Name"],
        "buyer_model":    data["models"]["buyer"],
        "seller_model":   data["models"]["seller"],
        "budget_scenario":data.get("budget_scenario"),
        "budget":         data.get("budget"),
        "retail_price":   retail,
        "wholesale_price":wholesale,
        "final_price":    final_price,
        "result":         result,
        "accepted":       accepted,
        "completed_turns":data.get("completed_turns", 0),
        "prr_buyer":      prr_buyer,
        "seller_profit":  seller_profit,
        "overpayment":    overpayment,
        "deadlock":       deadlock,
        "constraint_vio": constraint_vio,
    })

df = pd.DataFrame(records)
print(f"Loaded {len(df)} experiment records")
df.head()

In [ ]:
# ── Summary by model pair ────────────────────────────────────────────────────

summary = df.groupby(["buyer_model", "seller_model"]).agg(
    deal_rate      = ("accepted",       "mean"),
    avg_prr_buyer  = ("prr_buyer",      "mean"),
    avg_profit     = ("seller_profit",  "mean"),
    overpayment_rt = ("overpayment",    "mean"),
    deadlock_rt    = ("deadlock",       "mean"),
    constraint_rt  = ("constraint_vio", "mean"),
    n_experiments  = ("accepted",       "count"),
).reset_index()

summary["deal_rate"]      = summary["deal_rate"].map("{:.1%}".format)
summary["overpayment_rt"] = summary["overpayment_rt"].map("{:.1%}".format)
summary["deadlock_rt"]    = summary["deadlock_rt"].map("{:.1%}".format)
summary["constraint_rt"]  = summary["constraint_rt"].map("{:.1%}".format)

print(summary.to_string(index=False))

In [ ]:
# ── Deal Rate by Budget Scenario ─────────────────────────────────────────────

pivot = df.groupby(["budget_scenario", "buyer_model"])["accepted"].mean().unstack()

budget_order = ["high", "retail", "mid", "wholesale", "low"]
pivot = pivot.reindex([b for b in budget_order if b in pivot.index])

ax = pivot.plot(kind="bar", figsize=(10, 5), rot=0)
ax.set_title("Deal Rate by Budget Scenario and Buyer Model")
ax.set_xlabel("Budget Scenario")
ax.set_ylabel("Deal Rate")
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.legend(title="Buyer Model", bbox_to_anchor=(1.05, 1))
plt.tight_layout()
plt.savefig("deal_rate_by_budget.png", dpi=150)
plt.show()

In [ ]:
# ── Price Reduction Rate by Model Pair ──────────────────────────────────────

accepted_df = df[df["accepted"]].copy()

if len(accepted_df) > 0:
    pivot_prr = accepted_df.groupby(["buyer_model", "seller_model"])["prr_buyer"].mean().unstack()
    ax = pivot_prr.plot(kind="bar", figsize=(10, 5), rot=30)
    ax.set_title("Average Price Reduction Rate (%) by Model Pair")
    ax.set_xlabel("Buyer Model")
    ax.set_ylabel("Price Reduction Rate (%)")
    ax.legend(title="Seller Model", bbox_to_anchor=(1.05, 1))
    plt.tight_layout()
    plt.savefig("price_reduction_rate.png", dpi=150)
    plt.show()
else:
    print("No accepted deals yet — run more experiments first.")

In [ ]:
# ── Anomaly Summary ──────────────────────────────────────────────────────────

anomaly_cols = ["overpayment", "deadlock", "constraint_vio"]
anomaly_summary = df[anomaly_cols].mean() * 100

print("Anomaly rates across all experiments:")
for col, rate in anomaly_summary.items():
    print(f"  {col:20s}: {rate:.1f}%")

anomaly_summary.plot(kind="bar", figsize=(7, 4), rot=0, color=["#e74c3c", "#f39c12", "#9b59b6"])
plt.title("Anomaly Rates (%)")
plt.ylabel("%")
plt.tight_layout()
plt.savefig("anomaly_rates.png", dpi=150)
plt.show()